# **Telecomunicaciones: identificar operadores ineficaces**

In [1]:
# Cargar librerías
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from scipy import stats

In [2]:
# Cargar datos
clients = pd.read_csv('telecom_clients.csv')
calls = pd.read_csv('telecom_dataset_new.csv')
print(clients.head())
print("\n")
print(calls.head())

   user_id tariff_plan  date_start
0   166713           A  2019-08-15
1   166901           A  2019-08-23
2   168527           A  2019-10-29
3   167097           A  2019-09-01
4   168193           A  2019-10-16


   user_id                       date direction internal  operator_id  \
0   166377  2019-08-04 00:00:00+03:00        in    False          NaN   
1   166377  2019-08-05 00:00:00+03:00       out     True     880022.0   
2   166377  2019-08-05 00:00:00+03:00       out     True     880020.0   
3   166377  2019-08-05 00:00:00+03:00       out     True     880020.0   
4   166377  2019-08-05 00:00:00+03:00       out    False     880022.0   

   is_missed_call  calls_count  call_duration  total_call_duration  
0            True            2              0                    4  
1            True            3              0                    5  
2            True            1              0                    1  
3           False            1             10                   18  
4   

In [3]:
# Convertir fechas 
clients['date_start'] = pd.to_datetime(clients['date_start'])
calls['date'] = pd.to_datetime(calls['date'])

print(clients['date_start'].dtype)
print(calls['date'].dtype)
# Revisón de datos 
clients.info()
calls.info()

datetime64[ns]
datetime64[ns, pytz.FixedOffset(180)]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 732 entries, 0 to 731
Data columns (total 3 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   user_id      732 non-null    int64         
 1   tariff_plan  732 non-null    object        
 2   date_start   732 non-null    datetime64[ns]
dtypes: datetime64[ns](1), int64(1), object(1)
memory usage: 17.3+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53902 entries, 0 to 53901
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype                                
---  ------               --------------  -----                                
 0   user_id              53902 non-null  int64                                
 1   date                 53902 non-null  datetime64[ns, pytz.FixedOffset(180)]
 2   direction            53902 non-null  object                               
 3   internal       

In [4]:
# Revisión de datos ausentes 
clients.isna().sum()
calls.isna().sum()

user_id                   0
date                      0
direction                 0
internal                117
operator_id            8172
is_missed_call            0
calls_count               0
call_duration             0
total_call_duration       0
dtype: int64

In [5]:
# Revisar datos duplicados 
print("Duplicados de clients:", clients.duplicated().sum())
print("Duplicados de calls:", calls.duplicated().sum())

Duplicados de clients: 0
Duplicados de calls: 4900


In [6]:
# Verificar filas duplicadas
calls_duplicates = calls[calls.duplicated(keep=False)]
calls_duplicates.head(12)

,user_id,date,direction,internal,operator_id,is_missed_call,calls_count,call_duration,total_call_duration
6,166377,2019-08-05 00:00:00+03:00,out,False,880020.0,True,8,0,50
8,166377,2019-08-05 00:00:00+03:00,out,False,880020.0,True,8,0,50
27,166377,2019-08-12 00:00:00+03:00,in,False,NaN,True,2,0,34
28,166377,2019-08-12 00:00:00+03:00,in,False,NaN,True,2,0,34
38,166377,2019-08-14 00:00:00+03:00,in,False,NaN,True,1,0,3
43,166377,2019-08-14 00:00:00+03:00,out,False,880026.0,False,10,1567,1654
44,166377,2019-08-14 00:00:00+03:00,out,False,880026.0,False,10,1567,1654
45,166377,2019-08-14 00:00:00+03:00,in,False,NaN,True,1,0,3
46,166377,2019-08-15 00:00:00+03:00,out,False,880026.0,False,11,1413,1473
51,166377,2019-08-15 00:00:00+03:00,out,False,880026.0,False,11,1413,1473


In [7]:
# Eliminar operadores nulos de 'operator_id' e 'internal', convertir type
#calls_clean = calls.dropna(subset=['operator_id']).copy()
calls_clean = calls.copy()
calls_clean['operator_id'] = calls_clean['operator_id'].fillna(-1).astype(int)
calls_clean = calls_clean.dropna(subset=['internal'])
calls_clean.duplicated().sum()

# Eliminar duplicados 
calls_clean = calls_clean.drop_duplicates()
calls_clean.isna().sum()

print("\nDataset EDA (calls_clean) listo.")
print("Total filas:", calls_clean.shape[0])
print("Operator_id únicos:", calls_clean['operator_id'].nunique())
print("Llamadas sin operador (-1):", (calls_clean['operator_id'] == -1).sum())



Dataset EDA (calls_clean) listo.
Total filas: 48892
Operator_id únicos: 1093
Llamadas sin operador (-1): 7401


In [8]:
# Validar inconsistencias 
calls_clean['waiting_time'] = calls_clean['total_call_duration'] - calls_clean['call_duration']

print("Waiting time negativo:", (calls_clean['waiting_time'] < 0).sum())
print("Call duration > total:", (calls_clean['call_duration'] > calls_clean['total_call_duration']).sum())
print("Original:", len(calls))

calls_clean['waiting_time'].describe()

Waiting time negativo: 0
Call duration > total: 0
Original: 53902


count    48892.000000
mean       290.887671
std       1133.354018
min          0.000000
25%         17.000000
50%         55.000000
75%        200.000000
max      46474.000000
Name: waiting_time, dtype: float64

In [9]:
# Validaciones 
print("Valores nulos:") 
print(calls_clean.isna().sum())
print("\nValores duplicados:") 
print(calls_clean.duplicated().sum())
print("Llamadas salientes y entrantes:")
print(calls_clean['direction'].value_counts())
print("\nValores ocurrentes de internal:")
print(calls_clean['internal'].value_counts(dropna=False))
print("\nTotal de valores de calls_count:")
print(calls_clean['calls_count'].sum())
print("\nFue una llamada perdida:")
print(calls_clean['is_missed_call'].value_counts())
print("\n", calls_clean['calls_count'].describe())

Valores nulos:
user_id                0
date                   0
direction              0
internal               0
operator_id            0
is_missed_call         0
calls_count            0
call_duration          0
total_call_duration    0
waiting_time           0
dtype: int64

Valores duplicados:
0
Llamadas salientes y entrantes:
out    28997
in     19895
Name: direction, dtype: int64

Valores ocurrentes de internal:
False    43239
True      5653
Name: internal, dtype: int64

Total de valores de calls_count:
806484

Fue una llamada perdida:
False    27495
True     21397
Name: is_missed_call, dtype: int64

 count    48892.000000
mean        16.495214
std         63.671633
min          1.000000
25%          1.000000
50%          4.000000
75%         12.000000
max       4817.000000
Name: calls_count, dtype: float64


In [10]:
# Tabla cruzada para validar coherencia

pd.crosstab(calls_clean['is_missed_call'], calls_clean['call_duration'] == 0)

call_duration,False,True
is_missed_call,,
False,27478,17
True,295,21102


Se detectaron anomalías en la tabla de validación: 295 llamadas se marcaron como perdidas, pero la duración es mayor a 0; y 17 llamadas se marcaron como no perdidas, pero su duración es igual a 0.

In [11]:
calls_ops = calls_clean[calls_clean['operator_id'] != -1].copy()

print("\nDataset Operadores (calls_ops) listo.")
print("Total filas:", calls_ops.shape[0])
print("Operator_id únicos:", calls_ops['operator_id'].nunique())


Dataset Operadores (calls_ops) listo.
Total filas: 41491
Operator_id únicos: 1092


## **Análisis exploratorio de datos**

In [12]:
# Crear métricas por operador 
incoming = calls_ops[calls_ops['direction'] == 'in'].copy()
outgoing = calls_ops[calls_ops['direction'] == 'out'].copy()

# Métricas Incoming por calls_out
incoming_stats = incoming.groupby(['user_id', 'operator_id']).apply(
    lambda df: pd.Series({
        'incoming_calls': df['calls_count'].sum(),
        'missed_incoming_calls': df.loc[df['is_missed_call'] == True, 'calls_count'].sum(),
        'avg_waiting_time': np.average(df['waiting_time'], weights=df['calls_count']),
        'avg_call_duration_in': np.average(df['call_duration'], weights=df['calls_count'])
    })
).reset_index()

incoming_stats['missed_rate'] = incoming_stats['missed_incoming_calls'] / incoming_stats['incoming_calls']

# Métricas Outgoing por calls_count
outgoing_stats = outgoing.groupby(['user_id', 'operator_id']).apply(
    lambda df: pd.Series({
        'outgoing_calls': df['calls_count'].sum(),
        'avg_call_duration_out': np.average(df['call_duration'], weights=df['calls_count'])
    })
).reset_index()

# Combinar
operator_metrics = incoming_stats.merge(outgoing_stats, on=['user_id','operator_id'], how='outer')

# Rellenar NaNs
operator_metrics['incoming_calls'] = operator_metrics['incoming_calls'].fillna(0)
operator_metrics['missed_incoming_calls'] = operator_metrics['missed_incoming_calls'].fillna(0)
operator_metrics['avg_waiting_time'] = operator_metrics['avg_waiting_time'].fillna(0)
operator_metrics['outgoing_calls'] = operator_metrics['outgoing_calls'].fillna(0)
operator_metrics['missed_rate'] = operator_metrics['missed_rate'].fillna(0)

print(operator_metrics.head())

   user_id  operator_id  incoming_calls  missed_incoming_calls  \
0   166377       880020             7.0                    0.0   
1   166377       880022             8.0                    0.0   
2   166377       880026            24.0                    0.0   
3   166377       880028            63.0                    0.0   
4   166391       882476             3.0                    0.0   

   avg_waiting_time  avg_call_duration_in  missed_rate  outgoing_calls  \
0          7.714286             42.714286          0.0            38.0   
1         14.000000             64.000000          0.0           189.0   
2          9.666667            131.583333          0.0          2208.0   
3          9.761905            153.587302          0.0          2497.0   
4         31.666667             64.000000          0.0             0.0   

   avg_call_duration_out  
0             198.763158  
1             277.248677  
2            1531.754982  
3            1213.011614  
4                    Na

In [13]:
# Filtrar operadores con pocas llamadas
operator_metrics_filtered = operator_metrics[operator_metrics['incoming_calls'] >= 50].copy()

print("Operadores totales:", operator_metrics.shape[0])
print("Operadores con >=50 llamadas entrantes:", operator_metrics_filtered.shape[0])

Operadores totales: 1092
Operadores con >=50 llamadas entrantes: 234


In [15]:
# Definir umbrales de percentiles
missed_threshold = operator_metrics_filtered['missed_rate'].quantile(0.75)
waiting_threshold = operator_metrics_filtered['avg_waiting_time'].quantile(0.75)

# Para outgoing_calls: solo calcular percentil en operadores con outbound > 0
outgoing_nonzero = operator_metrics_filtered[operator_metrics_filtered['outgoing_calls'] > 0]

if len(outgoing_nonzero) > 0:
    outgoing_threshold = outgoing_nonzero['outgoing_calls'].quantile(0.25)
else:
    outgoing_threshold = 0

print("Umbrales:")
print(f"Missed_rate (75%): {missed_threshold:.3f}")
print(f"Avg_waiitng_time (75%): {waiting_threshold:.3f}")
print(f"Outgoing_calls (25%): {outgoing_threshold:.0f}")

Umbrales:
Missed_rate (75%): 0.016
Avg_waiitng_time (75%): 223.025
Outgoing_calls (25%): 113


In [16]:
# Clasificación de operadores ineficientes
operator_metrics_filtered.columns

operator_metrics_filtered['high_missed_rate'] = operator_metrics_filtered['missed_rate'] > missed_threshold
operator_metrics_filtered['high_waiting_time'] = operator_metrics_filtered['avg_waiting_time'] > waiting_threshold

# Low outbound cuando outgoing_calls > 0
operator_metrics_filtered['low_outgoing_calls'] = False
operator_metrics_filtered.loc[
    operator_metrics_filtered['outgoing_calls'] > 0, 'low_outgoing_calls'
] = operator_metrics_filtered.loc[
    operator_metrics_filtered['outgoing_calls'] > 0, 'outgoing_calls'
] < outgoing_threshold

operator_metrics_filtered['inefficiency_score'] = (
    operator_metrics_filtered['high_missed_rate'].astype(int) +
    operator_metrics_filtered['high_waiting_time'].astype(int) +
    operator_metrics_filtered['low_outgoing_calls'].astype(int)
)

# Clasificar operador como ineficiente si score >= 2
operator_metrics_filtered['is_inefficient'] = operator_metrics_filtered['inefficiency_score'] >=2

inefficient_operators = operator_metrics_filtered[operator_metrics_filtered['is_inefficient'] == True].copy()

print("Operadores ineficientes encontrados:", inefficient_operators.shape[0])

#print(operator_metrics_filtered.head())

Operadores ineficientes encontrados: 43


In [17]:
# Hipótesis estadísticas
efficient_group = operator_metrics_filtered[operator_metrics_filtered['is_inefficient'] == False]
inefficient_group = operator_metrics_filtered[operator_metrics_filtered['is_inefficient'] == True]

alpha = 0.05

# H1: ineficientes tienen mayor missed_rate
stat1, pval1 = stats.mannwhitneyu(
    inefficient_group['missed_rate'],
    efficient_group['missed_rate'],
    alternative = 'greater'
)

# H1: ineficientes tienen mayor waiting_time
stat2, pval2 = stats.mannwhitneyu(
    inefficient_group['avg_waiting_time'],
    efficient_group['avg_waiting_time'],
    alternative = 'greater'
)

# H1: ineficientes tienen menores outgoing_calls (solo para outbpund > 0)
efficient_out = efficient_group[efficient_group['outgoing_calls'] > 0]['outgoing_calls']
inefficient_out = inefficient_group[inefficient_group['outgoing_calls'] > 0]['outgoing_calls']

if len(efficient_out) > 0 and len(inefficient_out) > 0:
    stat3, pval3 = stats.mannwhitneyu(
        inefficient_out,
        efficient_out,
        alternative = 'less'
    )
else:
    stat3, pval3 = np.nan, np.nan

print("\nPRUEBAS DE HIPÓTESIS (alpha=0.05)")
print("\n1) Missed Rate (H1: ineficientes mayor missed_rate)")
print("p-value:", pval1, "| Rechazar H0?", pval1 < alpha)

print("\n2) Waiting Time (H1: ineficientes mayor waiting_time)")
print("p-value:", pval2, "| Rechazar H0?", pval2 < alpha)

print("\n3) Outgoing Calls (H1: ineficientes menos outgoing_calls, solo outbound>0)")
print("p-value:", pval3, "| Rechazar H0?", (pval3 < alpha) if not np.isnan(pval3) else "No se pudo evaluar")


PRUEBAS DE HIPÓTESIS (alpha=0.05)

1) Missed Rate (H1: ineficientes mayor missed_rate)
p-value: 3.664233953429214e-07 | Rechazar H0? True

2) Waiting Time (H1: ineficientes mayor waiting_time)
p-value: 4.4224259161935285e-10 | Rechazar H0? True

3) Outgoing Calls (H1: ineficientes menos outgoing_calls, solo outbound>0)
p-value: 6.673314134273454e-15 | Rechazar H0? True


In [18]:
# Top operadores ineficientes
print("Top 15 operadores ineficientes:")
print(
    inefficient_operators.sort_values(
        by=['inefficiency_score', 'missed_rate', 'avg_waiting_time'],
        ascending=False).head(15))

Top 15 operadores ineficientes:
     user_id  operator_id  incoming_calls  missed_incoming_calls  \
556   167977       944226           180.0                   30.0   
185   166916       906410           232.0                    7.0   
625   168155       938414            93.0                    2.0   
606   168091       958460           102.0                    2.0   
184   166916       906408           241.0                    4.0   
335   167200       905862           102.0                   15.0   
552   167977       944216           234.0                   24.0   
554   167977       944220           263.0                   18.0   
716   168307       945046            59.0                    4.0   
595   168062       951508           336.0                   21.0   
553   167977       944218           245.0                   15.0   
668   168187       948286           319.0                   15.0   
54    166536       900194            55.0                    2.0   
717   168307    

In [2]:
# Exportar archivos a csv
#calls_clean.to_csv("calls_clean_EDA.csv", index=False)                
#operator_metrics_filtered.to_csv("operator_metrics.csv", index=False)

print("Archivos exportados:")
print("- calls_all_EDA.csv")
print("- operator_metrics.csv")

Archivos exportados:
- calls_all_EDA.csv
- operator_metrics.csv
